# QDC Strategy Comparison and Backtest

Start with the [market data and performance notebook](qdc-etf-data-refresh.ipynb) to explore the instruments and propose rules. Section 5 here compares rebalancing in rounds 2–5 with holding after round 1. Section 9 runs several candidate rules on the same historical windows.

Run from this notebook's folder with Python 3.10 or later, pandas, and Matplotlib. Once the team chooses a configuration, enter it in `RULE` in the [trading workflow](qdc-etf-trading-workflow.ipynb). Both notebooks use the same instrument selection and order calculations.

## 1. Match the instrument list to prices

The ISINs come from the October 27, 2026 Belfius session materials. One ISIN can trade on multiple exchanges, so each instrument is mapped to a single EUR listing. We use Xetra for 46 instruments, Brussels for BEL 20, and Milan for 3× copper, 2× palladium, and 3× Nasdaq. EUR is the quote currency. Foreign holdings can still create currency exposure.

- The universe CSV stores the official number, ISIN, listing, and source. The original price file contains available daily closes, cash dividends per share, and split ratios from early 2025 to November 6, 2026. Later listings and unavailable quotes leave gaps.
- LYBK.DE split 80-for-1 on October 27, 2026. The provider restated the October 26 close early. We restore that close to its old share basis (€354.20) and multiply the shares held by 80 on the effective date.
- The unusual decline in 3HCL.MI on July 31, 2025 remains in the data. The issuer reported an intraday reset for the leveraged product that day.

Sources [Xetra instrument list](https://www.cashmarket.deutsche-boerse.com/cash-en/trading/Tradable-Instruments-Xetra/Downloads/xetra-downloads) · [Yahoo Finance historical quote example](https://finance.yahoo.com/quote/EUNL.DE/history/) · [Xetra split announcement](https://www.cashmarket.deutsche-boerse.com/cash-en/Stay-Informed/newsroom/Xetra-Frankfurt-Newsboard/XETR-CAPITAL-ADJUSTMENT-INFORMATION-27.10.2026-LU1829219390-5659892) · [3HCL issuer notice](https://www.wisdomtree.eu/-/media/eu-media-files/other-documents/operational/corp-action/boost/rns-corporate-actions/wtma---3hcl---restrike-event-july-2025.pdf?hash=227226A4F675A106C80594578DAE05EF&sc_lang=it-it)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from qdc_trading import load_market, market_metrics as shared_market_metrics, plan_round as shared_plan_round

data_dir = Path.cwd()
assets, close_all, close, dividend, split = load_market(data_dir)
print(f"{len(assets)} instruments and {len(close)} sessions with complete quotes")
equity_tickers = assets.index[assets["core_eligible"]]
print(f"Common quote range {close.index[0].date()} to {close.index[-1].date()}")
display(assets.groupby("venue").size().rename("Instrument count").to_frame())

## 2. Choose the historical windows

Use the rehearsal schedule. Trade on Monday and Thursday in each of the first two weeks and on Monday in week three. Value the portfolio at Wednesday's close in week three. Take the first eligible Monday per month and use non-overlapping windows. Choose dates based on coverage and schedule. Show the original October 19 window separately, now with 50 candidates.

Five daily returns need six closes. From the lookback period to the end of the window, any date with at least one quote must have all 50 quotes. Skip a start if a scheduled session is closed, quotes are incomplete, or a split falls on a trade date.

In [ ]:
starting_cash = 10_000.0
lookback = 5
cash_buffer = 0.01
max_orders = 5
round_offsets = pd.to_timedelta([0, 3, 7, 10, 14], unit="D")
end_offset = pd.Timedelta(days=16)

def eligible_window(start):
    trade_days = start + round_offsets
    end = start + end_offset
    past = close.index[close.index < start]
    if len(past) < lookback + 1 or not trade_days.isin(close.index).all() or end not in close.index:
        return False
    all_observed = close_all.loc[past[-lookback - 1]:end].index
    return all_observed.isin(close.index).all() and split.loc[trade_days].eq(1).all().all()


windows = []
seen_months = set()
for start in close.index[close.index.dayofweek == 0]:
    month = start.to_period("M")
    if month in seen_months or not eligible_window(start):
        continue
    assert not windows or start > windows[-1][1]
    windows.append((start + round_offsets, start + end_offset))
    seen_months.add(month)

assert len(windows) >= 3
original_start = pd.Timestamp("2026-10-19")
original_days = original_start + round_offsets
original_end = original_start + end_offset
assert eligible_window(original_start)
display(pd.DataFrame({
    "Monthly window start": [d[0].date() for d, _ in windows],
    "End": [end.date() for _, end in windows],
}))

## 3. Select instruments using the information available at the time

Use six closes before each trade date to compute five daily returns, accounting for dividends and share splits. Compound the returns to rank the core candidates. Multiply the standard deviation of daily returns by √252 to annualize volatility for the companion selection. Use maximum drawdown to inspect the steepest decline from a previous peak.

Pick the core from ordinary equity ETFs with the highest five-session total return. From the remaining 49 instruments, pick the companion with the lowest five-session volatility. The default allocates 70% and 30% of investable cash to these two positions. Other instruments have a target of zero shares.

These are the default decision rules. Section 9 compares other parameter combinations.

In [ ]:
def market_metrics(price_history, dividend_history, split_history):
    return shared_market_metrics(price_history, dividend_history, split_history).rename(
        columns={"累计收益": "Five-session return", "年化波动率": "Annualized volatility",
                 "最大回撤": "Maximum drawdown"}
    )


def plan_round(day, holdings, cash, **rule):
    parameters = {"cash_buffer": cash_buffer, **rule}
    return shared_plan_round(
        day, holdings, cash, close=close, dividend=dividend, split=split,
        equity_tickers=equity_tickers, **parameters,
    )

## 4. Turn targets into orders and carry the simulated holdings forward

Estimate the portfolio value using the previous close and the recorded holdings. Divide each target allocation by its reference price and round down to get the target shares. Subtract the existing holdings to get the order size. Positive shares are buys and negative shares are sells. Process sells first, sort orders on each side by descending estimated value, and keep up to five orders per round.

Simulate fills at the trade day's close. Skip an entire buy order if its execution would exceed the remaining cash. Credit dividends on the ex-dividend date and adjust share counts for splits. The simulation assumes immediate dividend credit and excludes commissions, slippage, and taxes.

In [ ]:
def execute_orders(orders, prices, holdings, cash):
    fills = []
    for ticker, order in orders.iterrows():
        quantity = int(order["delta"])
        price = prices[ticker]
        assert holdings[ticker] + quantity >= 0
        filled = quantity
        if quantity > 0 and quantity * price > cash:
            filled = 0
        holdings[ticker] += filled
        cash -= filled * price
        fills.append({
            "Ticker": ticker,
            "Planned shares": quantity,
            "Filled shares": filled,
            "Reference price": order["reference"],
            "Simulated fill price": price,
            "Cash after order": cash,
        })
    return cash, fills


def run_backtest(trade_days, end, rebalance_days, **rule):
    holdings = pd.Series(0, index=close.columns, dtype="int64")
    cash = starting_cash
    daily, rounds, orders_log = {}, [], []

    for day, prices in close.loc[trade_days[0]:end].iterrows():
        if day in rebalance_days:
            assert split.loc[day].eq(1).all()
            signal_day, core, partner, target, orders = plan_round(
                day, holdings, cash, **rule
            )
            cash_before = cash

        # Apply splits and dividends to existing positions before closing fills.
        new_units = holdings * split.loc[day]
        assert new_units.eq(new_units.round()).all()
        holdings = new_units.astype("int64")
        cash += (holdings * dividend.loc[day]).sum()

        if day in rebalance_days:
            cash, fills = execute_orders(orders, prices, holdings, cash)
            orders_log.extend({"Trade date": day, **fill} for fill in fills)
            rounds.append({
                "Trade date": day, "Signal cutoff": signal_day, "Core": core, "Companion": partner,
                "Target shares": target[target.ne(0)].to_dict(),
                "Planned orders": len(orders),
                "Filled orders": sum(fill["Filled shares"] != 0 for fill in fills),
                "Opening cash": cash_before, "Closing cash": cash,
                "Holdings": holdings[holdings.ne(0)].to_dict(),
            })

        market_value = (holdings * prices).sum()
        assert cash >= 0 and holdings.ge(0).all()
        daily[day] = {"Cash": cash, "Market value": market_value, "Portfolio value": cash + market_value}

    return pd.DataFrame.from_dict(daily, orient="index"), pd.DataFrame(rounds), pd.DataFrame(orders_log)

## 5. Change only whether rounds 2–5 rebalance

The buy-and-hold portfolio trades in round 1 only. Both portfolios start with identical fills, holdings, and cash, then both account for dividends and mark holdings to the daily close. The active strategy's advantage is the difference in final portfolio value. One basis point is 0.01 percentage points, or €1 on a €10,000 starting balance.

In [ ]:
def compare_window(trade_days, end):
    active, rounds, active_orders = run_backtest(trade_days, end, trade_days)
    hold, _, hold_orders = run_backtest(trade_days, end, trade_days[:1])
    assert active_orders.loc[active_orders["Trade date"].eq(trade_days[0])].equals(hold_orders)
    assert active.iloc[0].equals(hold.iloc[0])
    assert len(rounds) == 5 and rounds["Planned orders"].le(max_orders).all()
    assert rounds["Signal cutoff"].lt(rounds["Trade date"]).all()
    return pd.DataFrame({"Five rounds": active["Portfolio value"], "Buy and hold": hold["Portfolio value"]}), rounds, active_orders


def window_result(trade_days, end, equity):
    final = equity.iloc[-1]
    gains = final / starting_cash - 1
    return {
        "Start": trade_days[0].date(), "End": end.date(),
        "Five-round return": gains["Five rounds"],
        "Buy-and-hold return": gains["Buy and hold"],
        "Active advantage (EUR)": final["Five rounds"] - final["Buy and hold"],
        "Active advantage (bps)": (gains["Five rounds"] - gains["Buy and hold"]) * 10_000,
        "World equity price change": close.at[end, "EUNL.DE"] / close.at[trade_days[0], "EUNL.DE"] - 1,
    }


results = []
for trade_days, end in windows:
    equity, _, _ = compare_window(trade_days, end)
    results.append(window_result(trade_days, end, equity))

comparison = pd.DataFrame(results).set_index("Start")
edge = comparison["Active advantage (bps)"]
wins = edge.gt(1e-8).sum()
losses = edge.lt(-1e-8).sum()
ties = len(edge) - wins - losses
print(f"{len(edge)} windows | {wins} wins, {losses} losses, {ties} ties")
print(f"Mean advantage {edge.mean():+.1f} bps | median {edge.median():+.1f} bps")
print(f"Worst {edge.min():+.1f} bps | best {edge.max():+.1f} bps")
without_top_three = edge.nsmallest(len(edge) - 3)
print(f"Mean after excluding the three strongest windows {without_top_three.mean():+.1f} bps")
display(comparison.style.format({
    "Five-round return": "{:+.2%}", "Buy-and-hold return": "{:+.2%}",
    "Active advantage (EUR)": "{:+.2f}", "Active advantage (bps)": "{:+.1f}",
    "World equity price change": "{:+.2%}",
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(comparison.index, edge, width=14,
       color=["#167667" if value >= 0 else "#b5543c" for value in edge])
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set(ylabel="Active − buy & hold (bps)", title="Five-round strategy across historical windows")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 6. Inspect the original five-round rehearsal ledger

Run the same October 19 window with 50 eligible instruments. Inspect the first signal, round summaries, and individual orders. Positive shares are buys and negative shares are sells. A nonzero order with zero filled shares indicates that the simulation skipped it for insufficient cash. The chart shows the sum of cash and market value at each close.

In [ ]:
first_history = close.loc[close.index < original_start].tail(lookback + 1)
first_metrics = market_metrics(first_history, dividend.loc[first_history.index], split.loc[first_history.index])
print(f"The first round uses closes through {first_history.index[-1].date()}")
display(first_metrics.loc[equity_tickers].nlargest(3, "Five-session return").style.format("{:.2%}"))
display(first_metrics.nsmallest(3, "Annualized volatility").style.format("{:.2%}"))

original_equity, rounds, fills = compare_window(original_days, original_end)
original_result = window_result(original_days, original_end, original_equity)
print(f"50 candidates | Five rounds €{original_equity.iloc[-1]['Five rounds']:,.2f} | "
      f"Buy and hold €{original_equity.iloc[-1]['Buy and hold']:,.2f} | "
      f"Difference €{original_result['Active advantage (EUR)']:+.2f}")
display(rounds.style.format({"Opening cash": "{:.2f}", "Closing cash": "{:.2f}"}))
display(fills.style.format({"Reference price": "{:.4f}", "Simulated fill price": "{:.4f}", "Cash after order": "{:.2f}"}))

fig, ax = plt.subplots(figsize=(8.5, 3.5))
ax.plot(original_equity.index, original_equity["Five rounds"], marker="o", markersize=3, label="Five rounds")
ax.plot(original_equity.index, original_equity["Buy and hold"], linestyle="--", label="Buy & hold")
ax.set(ylabel="Portfolio value (EUR)", title="Original dates, expanded to 50 instruments")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 7. Interpret the results

Each monthly window starts with €10,000. The average advantage is the mean of the window-level differences. The broad equity price change gives market context. The comparison portfolio holds its round 1 positions.

The experiment measures how the five-session signal performed on these historical windows using today's universe. The sample is small and similar assets can move together. The simulation targets two holdings, uses the rehearsal calendar, and assumes zero trading costs. Confirm the official submission, timing, and execution rules before the competition.

In the initial November 6, 2026 snapshot, active rebalancing won 10 of 15 windows and lost 5. The average advantage was about €212 and fell to about €38 after excluding the three largest gains. The original three-instrument rehearsal gained about €1 relative to buy and hold. Expanding to 50 candidates over the same dates produced a shortfall of about €664. That rehearsal overlaps the October monthly window, so it appears separately.

**Optional AI support.** Give AI the window results and order logs to identify which trades and dates contributed to the difference. Ask it to cite the specific records for review.

## 8. Generate a sample order for one round

Use the latest local date with complete quotes for all 50 instruments. The example starts with €10,000 in cash and no holdings and allocates the investable amount 70/30. Use the [trading workflow](qdc-etf-trading-workflow.ipynb) for the full refresh, approval, and settlement process.

In [ ]:
issues_file = data_dir / "qdc-50-data-issues.csv"
if issues_file.is_file():
    refresh_issues = pd.read_csv(issues_file)
    print(f"The last refresh report contains {len(refresh_issues)} rows")
    if not refresh_issues.empty:
        display(refresh_issues)

demo_signal_day = close.index[-1]
demo_day = demo_signal_day + pd.Timedelta(days=1)
demo_cash = starting_cash
demo_holdings = pd.Series(0, index=close.columns, dtype="int64")
# Example: To start with 10 WORLD shares, set demo_holdings.loc["EUNL.DE"] = 10

skipped = close_all.index[close_all.index > demo_signal_day]
if len(skipped):
    print("The demo uses the last complete close. Quotes are incomplete on",
          ", ".join(day.strftime("%Y-%m-%d") for day in skipped))

signal_day, demo_core, demo_partner, demo_target, demo_orders = plan_round(
    demo_day, demo_holdings, demo_cash
)
assert signal_day == demo_signal_day < demo_day
assert len(demo_orders) <= max_orders

demo_history = close.loc[:signal_day].tail(lookback + 1)
demo_metrics = market_metrics(demo_history, dividend.loc[demo_history.index], split.loc[demo_history.index])
demo_table = pd.DataFrame({
    "Official label": assets.loc[demo_orders.index, "contest_label"],
    "Side": demo_orders["is_buy"].map({True: "Buy", False: "Sell"}),
    "Five-session return": demo_metrics.loc[demo_orders.index, "Five-session return"],
    "Annualized volatility": demo_metrics.loc[demo_orders.index, "Annualized volatility"],
    "Current shares": demo_holdings.loc[demo_orders.index],
    "Target shares": demo_target.loc[demo_orders.index],
    "Order shares": demo_orders["delta"].abs(),
    "Reference price (EUR)": demo_orders["reference"],
    "Estimated value (EUR)": demo_orders["reference_amount"],
})
demo_cash_after = demo_cash - (demo_orders["delta"] * demo_orders["reference"]).sum()
print(f"Signal cutoff {signal_day.date()} close | Assumed decision date {demo_day.date()}")
print(f"Core {demo_core} | Companion {demo_partner} | Estimated cash remaining €{demo_cash_after:,.2f}")
display(demo_table.style.format({
    "Five-session return": "{:+.2%}", "Annualized volatility": "{:.2%}",
    "Reference price (EUR)": "{:.4f}", "Estimated value (EUR)": "{:,.2f}",
}))

## 9. Compare multiple rules on the same windows

Add one entry to `RULES` for each candidate. `signal_sessions` is the lookback for selecting the core, `core_weight` is the core's share of the investable amount, and `cash_buffer` is the fraction reserved during order sizing. The companion still uses five-session volatility. The defaults cross 5 and 20 sessions with 70/30 and 50/50 allocations, all with a 1% cash reserve. Compare two rows that differ in one parameter to isolate that change.

The program keeps the section 2 windows that have enough complete quotes before round 1 for the longest lookback. Every rule uses the same dates, €10,000 starting capital, and simulated execution. A longer lookback can reduce the number of shared windows.

The summary reports average and median ending returns, average maximum drawdown, and how many windows beat the baseline. Drawdowns closer to zero are shallower. Use the window tables to see which dates drove the averages. The earlier and later groups show how results varied over time within this exploratory sample.

In [ ]:
RULES = {
    "5 sessions · 70/30": {"signal_sessions": 5, "core_weight": 0.70, "cash_buffer": 0.01},
    "5 sessions · 50/50": {"signal_sessions": 5, "core_weight": 0.50, "cash_buffer": 0.01},
    "20 sessions · 70/30": {"signal_sessions": 20, "core_weight": 0.70, "cash_buffer": 0.01},
    "20 sessions · 50/50": {"signal_sessions": 20, "core_weight": 0.50, "cash_buffer": 0.01},
}
REFERENCE_RULE = "5 sessions · 70/30"

longest_lookback = max(lookback, *(rule["signal_sessions"] for rule in RULES.values()))
shared_windows = []
for trade_days, end in windows:
    earlier = close.index[close.index < trade_days[0]]
    required_closes = longest_lookback + 1
    if len(earlier) < required_closes:
        continue
    observed = close_all.loc[earlier[-required_closes]:earlier[-1]].index
    if observed.isin(close.index).all():
        shared_windows.append((trade_days, end))

if not shared_windows:
    raise ValueError("Too few historical windows for these rules. Shorten the lookback or add more quotes.")
print(f"{len(RULES)} rules on the same {len(shared_windows)} historical windows.")
print(f"Baseline rule {REFERENCE_RULE}.")

In [ ]:
rows = []
for trade_days, end in shared_windows:
    for name, rule in RULES.items():
        ledger, _, _ = run_backtest(trade_days, end, trade_days, **rule)
        equity = ledger["Portfolio value"]
        peak = equity.cummax().clip(lower=starting_cash)
        rows.append({
            "Start": trade_days[0].date(),
            "Rule": name,
            "Ending return": equity.iloc[-1] / starting_cash - 1,
            "Maximum drawdown": (equity / peak - 1).min(),
        })

rule_results = pd.DataFrame(rows)
returns_by_window = rule_results.pivot(index="Start", columns="Rule", values="Ending return")
returns_by_window = returns_by_window.reindex(columns=RULES)
drawdowns_by_window = rule_results.pivot(index="Start", columns="Rule", values="Maximum drawdown")
drawdowns_by_window = drawdowns_by_window.reindex(columns=RULES)
edge_vs_reference = returns_by_window.sub(returns_by_window[REFERENCE_RULE], axis="index")
summary = pd.DataFrame({
    "Mean ending return": returns_by_window.mean(),
    "Median ending return": returns_by_window.median(),
    "Mean maximum drawdown": drawdowns_by_window.mean(),
    "Mean return difference vs. baseline": edge_vs_reference.mean(),
    "Windows above baseline": edge_vs_reference.gt(1e-8).sum(),
}).sort_values("Mean ending return", ascending=False)

cut = len(returns_by_window) * 2 // 3
if cut:
    summary[f"Mean return in first {cut} windows"] = returns_by_window.iloc[:cut].mean()
    summary[f"Mean return in last {len(returns_by_window) - cut} windows"] = returns_by_window.iloc[cut:].mean()

formats = {column: "{:+.2%}" for column in summary if column != "Windows above baseline"}
display(summary.style.format(formats).set_caption("Candidate rule summary"))
display(returns_by_window.style.format("{:+.2%}").set_caption("Ending returns by window"))
display(drawdowns_by_window.style.format("{:+.2%}").set_caption("Maximum drawdown by window"))

## 10. Record the rounds and review them after the competition

The table uses the rehearsal in section 6. Positive shares are buys and negative shares are sells. For the competition, record the suggested orders and the reviewer's decisions, then enter the fills reported by the organizer. The confirmed shares and prices determine cash and holdings.

| When | What to record |
|---|---|
| Before each round | Trade date, signal cutoff, rule version, opening cash and positions, and suggested shares |
| After review | Submitted tickers, sides, and shares, with the reasons for accepting, editing, or skipping orders |
| After the fill report | Confirmed prices and shares, fees or other cash adjustments, and closing cash and holdings |
| After the last round | Ending cash, market value, total return, maximum drawdown, and the rounds behind major differences |

Confirm the official execution-price convention before live use. Page 8 of the Belfius session materials describes both the previous session's close and the trade day's close. This rehearsal uses the trade day's close, as described in section 4.

In [ ]:
rehearsal_log = fills.merge(rounds[["Trade date", "Signal cutoff"]], on="Trade date", validate="many_to_one")
rehearsal_log = rehearsal_log.rename(columns={"Planned shares": "Proposed shares", "Filled shares": "Simulated filled shares"})
# The rehearsal submits the proposals. In live use, record the actual orders and review reasons.
rehearsal_log["Rehearsal submitted shares"] = rehearsal_log["Proposed shares"]
rehearsal_log["Review note"] = "Rehearsal submits the proposed orders"
rehearsal_log = rehearsal_log[[
    "Trade date", "Signal cutoff", "Ticker", "Proposed shares", "Rehearsal submitted shares", "Review note",
    "Simulated filled shares", "Simulated fill price", "Cash after order",
]]
display(rehearsal_log.style.format({"Simulated fill price": "{:.4f}", "Cash after order": "{:.2f}"}))